# `TRG_EMP` Full Load Conversion

**Source File:** `TRG_EMP_full_load.sql`
**Conversion Timestamp:** `2024-07-30T10:00:00Z`

This notebook performs a full load of employee data into the `TRG_EMP` target table from the `EMPLOYEES` source table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1001")
dbutils.widgets.text("ETL_PROC_WID", "5000")
dbutils.widgets.text("ODI_SESS_NO", "999999")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_job_type"))
display(spark.sql("SELECT * FROM v_datasource_num_id"))
display(spark.sql("SELECT * FROM v_etl_proc_wid"))
display(spark.sql("SELECT * FROM v_odi_sess_no"))

## Target Table: `workspace.hr.trg_emp`

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}: Perform full load into TRG_EMP
CREATE OR REPLACE TABLE workspace.hr.trg_emp
USING DELTA AS
SELECT
  EMPLOYEES.EMPLOYEE_ID AS EMPLOYEE_ID,
  EMPLOYEES.FIRST_NAME AS FIRST_NAME,
  EMPLOYEES.LAST_NAME AS LAST_NAME,
  EMPLOYEES.EMAIL AS EMAIL,
  EMPLOYEES.PHONE_NUMBER AS PHONE_NUMBER,
  EMPLOYEES.HIRE_DATE AS HIRE_DATE,
  EMPLOYEES.JOB_ID AS JOB_ID,
  EMPLOYEES.SALARY AS SALARY,
  EMPLOYEES.COMMISSION_PCT AS COMMISSION_PCT,
  EMPLOYEES.MANAGER_ID AS MANAGER_ID,
  EMPLOYEES.DEPARTMENT_ID AS DEPARTMENT_ID
FROM
  workspace.hr.employees AS EMPLOYEES;

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_emp;

## Optimize Target Table

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_emp ZORDER BY (DEPARTMENT_ID, EMPLOYEE_ID);

## Validation

In [ ]:
%sql
SELECT
  'Total Records' AS Metric,
  COUNT(*) AS Value
FROM workspace.hr.trg_emp;

In [ ]:
%sql
SELECT *
FROM workspace.hr.trg_emp
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Data Type Inference:** The `CREATE OR REPLACE TABLE` statement infers data types from the `workspace.hr.employees` source table. Verify these inferred types match the intended target schema and update manually if specific precision/scale or types are required (e.g., `DECIMAL(P,S)` instead of `DOUBLE`).
2.  **Schema Configuration:** Ensure the `workspace.hr` schema exists in your Databricks environment or update the schema path accordingly.
3.  **Source Table Existence:** The source table `workspace.hr.employees` must exist and be accessible.